<a href="https://colab.research.google.com/github/NUROISEA/anima-turbo-colab/blob/main/notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# @title Anima SD-CPP Install! { display-mode: "form" }

# @markdown **Run this first! :D**

# @markdown Your runtype **must** be T4 ***GPU*** or better. This will not run on CPU only. The binaries provided compiled without CPU support (?).

SD_CPP_LINK = "https://github.com/NUROISEA/sdcpp-google-colab-cuda-binary/releases/download/master-920-2f88688/sd-cli"
ANIMA_GGUF_LINK = "https://huggingface.co/vanes430/Anima-Turbo-V1.1-GGUF/resolve/main/anima-turbo-v1.1-Q8_0.gguf?download=true"
QWEN_VAE_LINK = "https://huggingface.co/circlestone-labs/Anima/resolve/main/split_files/vae/qwen_image_vae.safetensors?download=true"
QWEN_LLM_LINK = "https://huggingface.co/mradermacher/Qwen3-0.6B-Base-GGUF/resolve/main/Qwen3-0.6B-Base.Q8_0.gguf?download=true"

!wget -O sd-cli "{SD_CPP_LINK}"
!wget -O anima.gguf "{ANIMA_GGUF_LINK}"
!wget -O qwen_vae.safetensors "{QWEN_VAE_LINK}"
!wget -O qwen.gguf "{QWEN_LLM_LINK}"

!chmod +x ./sd-cli

# @markdown Click on `Show code` to see what's downloaded. While this runs, you queue to run the UI in the next cell.

--2026-09-27 11:16:30--  https://github.com/NUROISEA/sdcpp-google-colab-cuda-binary/releases/download/master-920-2f88688/sd-cli
Resolving github.com (github.com)... 140.82.113.4
Connecting to github.com (github.com)|140.82.113.4|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/1390158739/9ba70c4a-d7fb-435e-95c9-1b1f1443d127?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-09-27T12%3A05%3A45Z&rscd=attachment%3B+filename%3Dsd-cli&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&skt=2026-09-27T11%3A05%3A05Z&ske=2026-09-27T12%3A05%3A45Z&sks=b&skv=2018-11-09&sig=zZNNdnGpTIuY4hM%2FlmM6cgGxoWm9ZRwxkahDRRfSWlU%3D&jwt=eyJ0eXAiOiJKV1QiLCJhbGciOiJIUzI1NiJ9.eyJpc3MiOiJnaXRodWIuY29tIiwiYXVkIjoicmVsZWFzZS1hc3NldHMuZ2l0aHVidXNlcmNvbnRlbnQuY29tIiwia2V5Ijoia2V5MSIsImV4cCI6MTc5MDUwOTU5MSwibmJmIjoxNzkwNTA3NzkxLCJwYXRoIjoicmVsZWFzZWFzc2V0c

In [ ]:
# @title 🎨 Anima SD-CPP Image Generator { display-mode: "form" }

# @markdown **Then this one, :)**

# @markdown **Note:** ✨ *This is vibecoded to shit* ✨

# @markdown You can connect your google drive to this one, just look at the outputs section on the UI.

import glob
import html
import os
import random
import shlex
import shutil
import subprocess
from datetime import datetime
from functools import partial

import ipywidgets as widgets
from IPython.display import clear_output, display

# ---------------------------------------------------------------------------
# 0. LAYOUT-ONLY CSS + THEME-INHERIT (no palette: resize handles,
#    wrapping helpers, and Colab light/dark text inheritance)
# ---------------------------------------------------------------------------
# textarea -> native vertical drag-resize.
# .image-resize-box -> drag-resizable preview container.
# Theme part only inherits Jupyter's own label color -- no hardcoded colors,
# so Collapse/Accordion headers stay readable in both light and dark mode.
# (Selectors verified in Colab devtools: phosphor .p-Widget/.lm-Widget
#  + .p-Collapse-header, color yoinked as --jp-widgets-label-color.)
display(
    widgets.HTML(
        "<style>"
        ":root { color-scheme: light dark; }"
        "textarea { resize: vertical !important; }"
        ".image-resize-box { resize: both !important; "
        "overflow: auto !important; "
        "min-width: 240px; min-height: 240px; }"
        ".lm-Widget, .p-Widget, .p-Collapse-header"
        "{ color: var(--jp-widgets-label-color, inherit) !important; }"
        "</style>"
    )
)

# ---------------------------------------------------------------------------
# 1. UI CONTROL DEFINITIONS (LEFT COLUMN)
# ---------------------------------------------------------------------------

prompt_w = widgets.Textarea(
    value="year 2025, score_5, best quality, absurdres, safe, ",
    description="Prompt:",
    layout=widgets.Layout(width="100%", height="120px"),
)

negative_w = widgets.Textarea(
    value="nsfw, questionable, explicit, worst quality, low quality, score_1, score_2, score_3, artist name, blurry, jpeg artifacts, chromatic aberration",
    description="Negative:",
    layout=widgets.Layout(width="100%", height="80px"),
)

width_w = widgets.IntSlider(
    value=832, min=64, max=2048, step=64,
    description="Width:", continuous_update=False,
)
height_w = widgets.IntSlider(
    value=1216, min=64, max=2048, step=64,
    description="Height:", continuous_update=False,
)
steps_w = widgets.IntSlider(
    value=8, min=1, max=150, step=1,
    description="Steps:", continuous_update=False,
)
cfg_w = widgets.FloatSlider(
    value=1.0, min=1.0, max=30.0, step=0.5,
    description="CFG Scale:", continuous_update=False,
)
seed_w = widgets.IntText(value=-1, description="Seed:")
sampler_w = widgets.Dropdown(
    options=["euler", "euler_a", "heun", "dpm2", "dpm++2s_a",
               "dpm++2m", "dpm++2mv2", "ipndm", "ipndm_v", "lcm",
               "ddim_trailing", "tcd", "res_multistep", "res_2s",
               "er_sde", "euler_cfg_pp", "euler_a_cfg_pp", "euler_ge",
               "dpm++2m_sde", "dpm++2m_sde_bt", "lms"],
    value="euler",
    description="Sampler:",
)
scheduler_w = widgets.Dropdown(
    options=["auto (default)", "discrete", "karras", "exponential",
               "ays", "gits", "sgm_uniform", "simple", "smoothstep",
               "kl_optimal", "lcm", "bong_tangent", "ltx2",
               "logit_normal", "flux2", "flux", "beta", "llada_image"],
    value="auto (default)",
    description="Scheduler:",
)
rng_w = widgets.Dropdown(
    options=["cuda", "cpu", "std_default"],
    value="cuda",
    description="RNG:",
)
batch_w = widgets.IntSlider(
    value=1, min=1, max=8, step=1,
    description="Batch:", continuous_update=False,
)
gen_count_w = widgets.IntSlider(
    value=1, min=1, max=8, step=1,
    description="Gen count:", continuous_update=False,
)
total_lbl = widgets.HTML("<small>Total images: 1</small>")


def update_total_lbl(_c=None):
    total_lbl.value = (
        f"<small>Total images: {batch_w.value * gen_count_w.value} "
        f"({gen_count_w.value} gen x {batch_w.value} batch)</small>"
    )


batch_w.observe(update_total_lbl, names="value")
gen_count_w.observe(update_total_lbl, names="value")

fa_w = widgets.Checkbox(value=True, description="Flash Attention")
verbose_w = widgets.Checkbox(value=False, description="Verbose")
guidance_w = widgets.FloatSlider(
    value=3.5, min=0.0, max=20.0, step=0.5,
    description="Guidance:", continuous_update=False,
)
clip_skip_w = widgets.IntSlider(
    value=-1, min=-1, max=12, step=1,
    description="Clip skip:", continuous_update=False,
)
advanced_box = widgets.Accordion(
    children=[widgets.VBox([
        scheduler_w, rng_w, guidance_w, clip_skip_w,
        widgets.HBox([fa_w, verbose_w]),
    ])],
)
advanced_box.set_title(0, "Advanced")

# Tier 3 model paths: tucked away, collapsed by default.
diffusion_model_w = widgets.Text(
    value="anima.gguf", description="Diffusion:",
    layout=widgets.Layout(width="100%"),
)
vae_model_w = widgets.Text(
    value="qwen_vae.safetensors", description="VAE:",
    layout=widgets.Layout(width="100%"),
)
llm_model_w = widgets.Text(
    value="qwen.gguf", description="LLM:",
    layout=widgets.Layout(width="100%"),
)
backend_w = widgets.Text(
    value="diffusion=cuda0,te=cuda0,vae=cuda0",
    description="Backend:",
    layout=widgets.Layout(width="100%"),
)
models_box = widgets.Accordion(
    children=[widgets.VBox(
        [diffusion_model_w, vae_model_w, llm_model_w, backend_w]
    )],
)
models_box.set_title(0, "Models (advanced)")

output_dir_w = widgets.Text(
    value="outputs",
    description="Output Dir:",
    layout=widgets.Layout(width="100%"),
)
drive_dir_w = widgets.Text(
    value="/content/drive/MyDrive/sd-outputs",
    description="Drive Dir:",
    layout=widgets.Layout(width="100%"),
)
mount_drive_btn = widgets.Button(
    description="🔌 Mount Drive",
    layout=widgets.Layout(width="100%"),
)
sync_drive_btn = widgets.Button(
    description="☁️ Copy to Drive",
    layout=widgets.Layout(width="50%"),
)
pull_drive_btn = widgets.Button(
    description="⬇️ Copy from Drive",
    layout=widgets.Layout(width="50%"),
)
auto_sync_w = widgets.Checkbox(
    value=False, description="Auto-copy to Drive",
)
drive_status_lbl = widgets.HTML("<small>Drive not synced.</small>")
outputs_box = widgets.Accordion(
    children=[widgets.VBox([
        output_dir_w,
        drive_dir_w,
        mount_drive_btn,
        widgets.HBox([sync_drive_btn,pull_drive_btn,]),
        auto_sync_w,
        drive_status_lbl,
    ])],
)
outputs_box.set_title(0, "Outputs folder")

generate_btn = widgets.Button(
    description="🚀 Generate Image",
    button_style="success",
    layout=widgets.Layout(width="100%", height="45px"),
)

log_output = widgets.Output(
    layout=widgets.Layout(
        width="100%", height="200px", overflow_y="auto",
    )
)


def mount_drive(_b=None):
    with log_output:
        try:
            from google.colab import drive
        except ImportError:
            print("Google Drive mount is only available in Colab.")
            return
        drive.mount("/content/drive")
        print("Drive mounted at /content/drive")
    drive_status_lbl.value = "<small>Drive mounted.</small>"


def do_copy_to_drive():
    src = output_dir_w.value
    dst = drive_dir_w.value
    if not os.path.isdir(src):
        return 0, src, dst, f"{src} not found."
    os.makedirs(dst, exist_ok=True)
    copied = 0
    for ext in ("*.png", "*.jpg", "*.jpeg", "*.webp", "*.txt"):
        for path in glob.glob(os.path.join(src, ext)):
            shutil.copy2(path, os.path.join(dst, os.path.basename(path)))
            copied += 1
    return copied, src, dst, None


def sync_to_drive(_b=None):
    copied, src, dst, err = do_copy_to_drive()
    if err:
        drive_status_lbl.value = f"<small>Nothing to copy: {err}</small>"
        return
    drive_status_lbl.value = f"<small>Copied {copied} files to " \
        f"{dst}.</small>"
    with log_output:
        print(f"☁️ Copied {copied} files: {src} -> {dst}")


mount_drive_btn.on_click(mount_drive)
sync_drive_btn.on_click(sync_to_drive)
pull_drive_btn.on_click(lambda _b: pull_from_drive())


def do_copy_from_drive():
    src = drive_dir_w.value
    dst = output_dir_w.value
    if not os.path.isdir(src):
        return 0, src, dst, f"{src} not found."
    os.makedirs(dst, exist_ok=True)
    copied = 0
    for ext in ("*.png", "*.jpg", "*.jpeg", "*.webp", "*.txt"):
        for path in glob.glob(os.path.join(src, ext)):
            target = os.path.join(dst, os.path.basename(path))
            if os.path.exists(target):
                continue
            shutil.copy2(path, target)
            copied += 1
    return copied, src, dst, None


def pull_from_drive():
    copied, src, dst, err = do_copy_from_drive()
    if err:
        drive_status_lbl.value = f"<small>Nothing to pull: {err}</small>"
        return
    drive_status_lbl.value = f"<small>Pulled {copied} new files to " \
        f"{dst}.</small>"
    with log_output:
        print(f"⬇️ Pulled {copied} new files: {src} -> {dst}")
    history_page["page"] = 0
    refresh_history()

# ---------------------------------------------------------------------------
# 2. CENTER: IMAGE PREVIEW + PREVIEW-HEIGHT SLIDER
# ---------------------------------------------------------------------------

image_display = widgets.Image(
    layout=widgets.Layout(
        width="100%", max_width="100%",
        max_height="800px", object_fit="contain",
    )
)

preview_h_w = widgets.IntSlider(
    value=800, min=300, max=1200, step=50,
    description="Preview H:",
    continuous_update=False,
    layout=widgets.Layout(width="100%"),
)


def on_preview_h_change(change):
    image_display.layout.max_height = f"{change['new']}px"


preview_h_w.observe(on_preview_h_change, names="value")

# Resizable wrapper: drag the corner to resize the preview pane.
image_box = widgets.Box(
    [image_display],
    layout=widgets.Layout(
        width="100%", overflow="auto",
    ),
)
image_box.add_class("image-resize-box")

image_info_lbl = widgets.HTML("<small>No image yet.</small>")
gen_info_lbl = widgets.HTML("<small>Generation info will appear here.</small>")


def read_settings_sidecar(image_path):
    txt_path = os.path.splitext(image_path)[0] + ".txt"
    if not os.path.exists(txt_path):
        return None
    try:
        settings = {}
        with open(txt_path, encoding="utf-8") as f:
            for line in f:
                if ":" not in line:
                    continue
                key, val = line.split(":", 1)
                settings[key.strip()] = val.strip()
        return settings
    except OSError:
        return None


def read_seed_from_sidecar(image_path):
    settings = read_settings_sidecar(image_path)
    if not settings:
        return None
    return settings.get("seed")


def format_gen_info_html(settings):
    if not settings:
        return "<small>No settings file found for this image.</small>"
    prompt = html.escape(settings.get("prompt", "?"))
    negative = html.escape(settings.get("negative_prompt", "?"))
    rows = [
        ("Size", f"{settings.get('width', '?')}x" \
         f"{settings.get('height', '?')}"),
        ("Steps", settings.get("steps", "?")),
        ("CFG", settings.get("cfg_scale", "?")),
        ("Guidance", settings.get("guidance", "?")),
        ("Seed", settings.get("seed", "?")),
        ("Sampler", settings.get("sampler", "?")),
        ("Scheduler", settings.get("scheduler", "?")),
        ("RNG", settings.get("rng", "?")),
        ("Clip skip", settings.get("clip_skip", "?")),
    ]
    summary = " · ".join(f"{k}: {html.escape(str(v))}" for k, v in rows)
    return (
        f"<small>{summary}</small>"
        f"<details><summary><small>Prompt / Negative</small></summary>"
        f"<small><b>Prompt:</b> {prompt}<br>"
        f"<b>Negative:</b> {negative}</small></details>"
    )


def set_preview(image_path, seed=None):
    settings = read_settings_sidecar(image_path)
    if seed is None:
        seed = settings.get("seed") if settings else None
    with open(image_path, "rb") as f:
        image_display.value = f.read()
    name = os.path.basename(image_path)
    seed_txt = seed if seed is not None else "unknown"
    image_info_lbl.value = f"<small><b>{html.escape(name)}</b> — " \
        f"seed: {html.escape(str(seed_txt))}</small>"
    gen_info_lbl.value = format_gen_info_html(settings)

# ---------------------------------------------------------------------------
# 3. RIGHT: HISTORY GALLERY
# ---------------------------------------------------------------------------

history_page = {"page": 0}
HISTORY_PAGE_SIZE = 10

history_box = widgets.VBox(
    [],
    layout=widgets.Layout(width="100%", overflow_y="auto"),
)
refresh_btn = widgets.Button(
    description="↻ Refresh",
    layout=widgets.Layout(width="100%"),
)
prev_btn = widgets.Button(
    description="◀ Prev",
    layout=widgets.Layout(width="50%"),
)
next_btn = widgets.Button(
    description="Next ▶",
    layout=widgets.Layout(width="50%"),
)
history_page_lbl = widgets.HTML("<small>Page 1</small>")


def list_saved_images(out_dir):
    if not out_dir or not os.path.isdir(out_dir):
        return []
    files = []
    for ext in ("*.png", "*.jpg", "*.jpeg", "*.webp"):
        files.extend(glob.glob(os.path.join(out_dir, ext)))
    files.sort(key=os.path.getmtime, reverse=True)
    return files


def load_history_image(path, _b=None):
    if not os.path.exists(path):
        return
    set_preview(path)


def refresh_history(_b=None):
    paths = list_saved_images(output_dir_w.value)
    total = len(paths)
    pages = max(1, (total + HISTORY_PAGE_SIZE - 1) // HISTORY_PAGE_SIZE)
    history_page["page"] = max(0, min(history_page["page"], pages - 1))
    page = history_page["page"]
    chunk = paths[page * HISTORY_PAGE_SIZE:(page + 1) * HISTORY_PAGE_SIZE]
    rows = []
    if not chunk:
        rows.append(widgets.HTML("<i>No images yet.</i>"))
    for path in chunk:
        try:
            with open(path, "rb") as f:
                thumb = widgets.Image(
                    value=f.read(),
                    layout=widgets.Layout(width="64px", height="64px"),
                )
        except OSError:
            continue
        name = os.path.basename(path)
        load_btn = widgets.Button(
            description=name,
            tooltip=path,
            layout=widgets.Layout(width="100%"),
        )
        load_btn.on_click(partial(load_history_image, path))
        rows.append(
            widgets.HBox(
                [thumb, load_btn],
                layout=widgets.Layout(width="100%"),
            )
        )
    history_box.children = tuple(rows)
    history_page_lbl.value = f"<small>Page {page + 1}/{pages} " \
        f"({total} images)</small>"
    prev_btn.disabled = page <= 0
    next_btn.disabled = page >= pages - 1


def go_prev(_b=None):
    history_page["page"] = max(0, history_page["page"] - 1)
    refresh_history()


def go_next(_b=None):
    history_page["page"] += 1
    refresh_history()


prev_btn.on_click(go_prev)
next_btn.on_click(go_next)
refresh_btn.on_click(refresh_history)
output_dir_w.observe(lambda _c: (history_page.update(page=0),
                                 refresh_history()), names="value")

# ---------------------------------------------------------------------------
# 4. GENERATION HANDLER
# ---------------------------------------------------------------------------


def write_settings_sidecar(path, seed, timestamp):
    with open(os.path.splitext(path)[0] + ".txt", "w",
              encoding="utf-8") as f:
        f.write(f"prompt: {prompt_w.value}\n")
        f.write(f"negative_prompt: {negative_w.value}\n")
        f.write(f"width: {width_w.value}\n")
        f.write(f"height: {height_w.value}\n")
        f.write(f"steps: {steps_w.value}\n")
        f.write(f"cfg_scale: {cfg_w.value}\n")
        f.write(f"guidance: {guidance_w.value}\n")
        f.write(f"seed: {seed}\n")
        f.write(f"sampler: {sampler_w.value}\n")
        f.write(f"scheduler: {scheduler_w.value}\n")
        f.write(f"rng: {rng_w.value}\n")
        f.write(f"batch_count: {batch_w.value}\n")
        f.write(f"clip_skip: {clip_skip_w.value}\n")
        f.write(f"diffusion_model: {diffusion_model_w.value}\n")
        f.write(f"vae: {vae_model_w.value}\n")
        f.write(f"llm: {llm_model_w.value}\n")
        f.write(f"backend: {backend_w.value}\n")
        f.write(f"diffusion_fa: {fa_w.value}\n")
        f.write(f"output: {path}\n")
        f.write(f"created: {timestamp}\n")


def generate_image(b):
    batch = batch_w.value
    gens = gen_count_w.value
    total = batch * gens
    base_random = seed_w.value < 0
    base_seed = seed_w.value
    out_dir = output_dir_w.value
    os.makedirs(out_dir, exist_ok=True)
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

    generate_btn.disabled = True
    saved_paths = []

    with log_output:
        clear_output()
        next_fixed_seed = base_seed
        used_random_seeds = set()
        for gi in range(gens):
            if base_random:
                gen_seed = random.randint(0, 2**31 - 1)
                while gen_seed in used_random_seeds:
                    gen_seed = random.randint(0, 2**31 - 1)
            else:
                # Reserve a contiguous block of `batch` seeds per gen,
                # so gen blocks can never overlap by construction.
                gen_seed = next_fixed_seed % (2**31)
                next_fixed_seed += batch
            generate_btn.description = f"⏳ Gen {gi + 1}/{gens}..."
            print(f"=== Gen {gi + 1}/{gens} (seed {gen_seed}) ===")
            if total == 1:
                out_arg = os.path.join(out_dir, f"output_{timestamp}.png")
                expect_glob = [out_arg]
            else:
                stem = f"output_{timestamp}_g{gi + 1:02d}_%03d.png"
                out_arg = os.path.join(out_dir, stem)
                prefix = stem.split("%03d")[0]
                expect_glob = sorted(
                    glob.glob(os.path.join(out_dir, prefix + "*.png"))
                )
                # snapshot before, so we only pick up new files
                before = set(expect_glob)
            args = [
                "./sd-cli",
                "--diffusion-model", diffusion_model_w.value,
                "--vae", vae_model_w.value,
                "--llm", llm_model_w.value,
                "-p", prompt_w.value,
                "--negative-prompt", negative_w.value,
                "--height", str(height_w.value),
                "--width", str(width_w.value),
                "--steps", str(steps_w.value),
                "--cfg-scale", str(cfg_w.value),
                "--guidance", str(guidance_w.value),
                "--seed", str(gen_seed),
                "--sampling-method", sampler_w.value,
                "--rng", rng_w.value,
                "--backend", backend_w.value,
                "-o", out_arg,
            ]
            if scheduler_w.value != "auto (default)":
                args += ["--scheduler", scheduler_w.value]
            if clip_skip_w.value >= 0:
                args += ["--clip-skip", str(clip_skip_w.value)]
            if batch > 1:
                args += ["--batch-count", str(batch)]
            if fa_w.value:
                args.append("--diffusion-fa")
            if verbose_w.value:
                args.append("-v")
            print(" ".join(shlex.quote(a) for a in args))
            print("-" * 50)
            process = subprocess.Popen(
                args, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                text=True,
            )
            for line in process.stdout:
                print(line, end="")
            process.wait()
            if total == 1:
                new_files = [out_arg] if os.path.exists(out_arg) else []
            else:
                prefix = stem.split("%03d")[0]
                after = sorted(
                    glob.glob(os.path.join(out_dir, prefix + "*.png"))
                )
                new_files = [p for p in after if p not in before]
                if not new_files:
                    # fallback: sd-cli may start indexing at 1
                    new_files = [p for p in after if p not in saved_paths]
            if not new_files:
                print(f"\n❌ Gen {gi + 1} produced no files.")
                continue
            for bi, path in enumerate(sorted(new_files)):
                # sd-cli batches from one seed; assume +1 per image.
                img_seed = (gen_seed + bi) % (2**31)
                if base_random:
                    used_random_seeds.add(img_seed)
                print(f"\n✨ Saved to: {path} (seed {img_seed})")
                write_settings_sidecar(path, img_seed, timestamp)
                saved_paths.append(path)
                set_preview(path, str(img_seed))
        print(f"\nDone: {len(saved_paths)}/{total} images.")
        if auto_sync_w.value and saved_paths:
            copied, src, dst, err = do_copy_to_drive()
            if err:
                print(f"☁️ Auto-copy skipped: {err}")
            else:
                print(f"☁️ Auto-copied {copied} files: {src} -> {dst}")
                drive_status_lbl.value = f"<small>Copied {copied} " \
                    f"files to {dst}.</small>"
    history_page["page"] = 0
    refresh_history()
    generate_btn.disabled = False
    generate_btn.description = "🚀 Generate Image"


generate_btn.on_click(generate_image)

# ---------------------------------------------------------------------------
# 5. THREE-COLUMN RESPONSIVE LAYOUT
# ---------------------------------------------------------------------------
# flex + min_width + row wrap = side-by-side on 1080p, tighter on
# 768p laptop, stacked on phone. Center gets 2x grow factor.
wrap_row = widgets.Layout(width="100%", flex_flow="row wrap")

controls_col = widgets.VBox(
    [
        widgets.HTML("<h3>📝 Parameters</h3>"),
        prompt_w,
        negative_w,
        widgets.HBox([width_w, height_w], layout=wrap_row),
        widgets.HBox([steps_w, cfg_w], layout=wrap_row),
        widgets.HBox([seed_w, sampler_w], layout=wrap_row),
        widgets.HBox([batch_w, gen_count_w], layout=wrap_row),
        total_lbl,
        advanced_box,
        models_box,
        outputs_box,
        widgets.Box(layout=widgets.Layout(height="10px")),
        generate_btn,
        widgets.HTML("<b>Console Logs:</b>"),
        log_output,
    ],
    layout=widgets.Layout(
        flex="1 1 300px", min_width="280px", max_width="520px",
        padding="0 12px 0 0",
    ),
)

image_col = widgets.VBox(
    [
        widgets.HTML("<h3>🖼️ Output Preview</h3>"),
        image_info_lbl,
        preview_h_w,
        image_box,
        widgets.HTML("<b>Generation info:</b>"),
        gen_info_lbl,
        widgets.HTML(
            "<small>Tip: drag the corner of the preview "
            "to resize; use Preview H slider for height.</small>"
        ),
    ],
    layout=widgets.Layout(
        flex="2 1 380px", min_width="300px", padding="0 12px",
    ),
)

history_col = widgets.VBox(
    [
        widgets.HTML("<h3>🕘 History</h3>"),
        refresh_btn,
        history_page_lbl,
        widgets.HBox([prev_btn, next_btn]),
        history_box,
    ],
    layout=widgets.Layout(
        flex="1 1 220px", min_width="200px", max_width="320px",
        padding="0 0 0 12px",
    ),
)

main_row = widgets.HBox(
    [controls_col, image_col, history_col],
    layout=widgets.Layout(
        width="100%", flex_flow="row wrap", align_items="flex-start",
    ),
)

ui = widgets.VBox(
    [
        widgets.HTML("<h2>🎨 Anima SD-CPP Studio</h2>"),
        main_row,
    ]
)

refresh_history()
display(ui)

HTML(value='<style>:root { color-scheme: light dark; }textarea { resize: vertical !important; }.image-resize-b…